# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitttt077/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Chosen Lane: Lane 2 — Refresh / Content Opportunity Scoring**

Content and SEO teams face an asymmetric capacity bottleneck: an enterprise site often maintains 10,000 to 50,000 published pages, yet editorial teams have the bandwidth to thoroughly audit, update, and rewrite only 20 to 50 pages per month. Today, most organizations rely on simplistic, static heuristics (e.g., "refresh any page older than 6 months" or "sort by lowest traffic drop in Google Analytics"). These manual rules waste precious editorial hours updating low-intent, dormant articles while missing severe, compounding traffic decay on high-visibility pillar content.

I selected Lane 2 because it tackles an urgent, high-stakes operational decision: transforming an unmanageable ocean of published content into an explainable, prioritized action queue. By learning to detect multi-signal patterns of decay across freshness, SERP positioning, CTR, and search intent, a learned ranking model can multiply the effectiveness of human editorial labor.

In [1]:
import os
import pandas as pd
import numpy as np

# Adjust working directory if running inside work/notebooks/
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_pages = len(df)
declining_pages = df["trend_direction"].eq("down").sum()
decline_rate = declining_pages / total_pages

print(f"Total analyzed pages: {total_pages:,}")
print(f"Actively declining pages ('trend_direction == down'): {declining_pages:,} ({decline_rate * 100:.1f}%)")
print("\nObserved Trend Direction Distribution:")
print(df["trend_direction"].value_counts().to_string())
print(f"\nEditorial triage challenge: With {declining_pages:,} decaying assets and an editorial capacity")
print("of ~50 pages/month, human reviewers can inspect less than 0.3% of declining content.")
print("Prioritization quality determines whether traffic recovers or continues compounding decay.")

Total analyzed pages: 30,000
Actively declining pages ('trend_direction == down'): 16,262 (54.2%)

Observed Trend Direction Distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Editorial triage challenge: With 16,262 decaying assets and an editorial capacity
of ~50 pages/month, human reviewers can inspect less than 0.3% of declining content.
Prioritization quality determines whether traffic recovers or continues compounding decay.


## 2. The question: decision, action, cost of a wrong call

### The Core Decision
*Which top-50 content assets should an SEO editor or content marketing lead prioritize for an in-depth refresh this review cycle?*

### Who Acts on the Output & What They Do
- **Actor**: Content Strategist / Senior SEO Editor.
- **Action Taken**: Pulls the top-ranked candidates from the priority queue and executes targeted intervention:
  1. Audit keyword intent mismatch and update outdated factual claims.
  2. Refresh title tags, headers, and meta descriptions to improve CTR.
  3. Expand thin sections or consolidate cannibalizing articles.
  4. Implement structured data schema or schedule for pruning/redirect.

### The Cost of a Wrong Call
- **False Positive (Flagging a page that doesn't need refresh or has no recovery potential)**:
  - Wastes 3 to 6 hours of skilled editorial labor per article ($150–$300 direct labor cost).
  - High opportunity cost: editing stable, self-sustaining content diverts bandwidth from actual decaying assets.
- **False Negative (Failing to flag a high-traffic pillar asset in steep decline)**:
  - Page slips from Position 2–3 down to page 2, losing 70–90% of its organic impressions and click-throughs.
  - Compounding search revenue and lead generation losses that require months of expensive link building or total rewrites to regain.

### Operational Metric Alignment
Because reviewers inspect a fixed batch of pages per cycle, **`Precision@50`** (and `Precision@20`) is the governing operational metric. Generic accuracy or ROC-AUC across 30,000 pages is irrelevant if the top 50 picks are filled with unfixable or healthy pages.

In [2]:
# Quantifying the high-stakes editorial sub-population
high_visibility = df[df["impressions_90d"] >= 500]
high_vis_decay = high_visibility["trend_direction"].eq("down").sum()
high_vis_decay_rate = high_vis_decay / len(high_visibility)

print(f"High-visibility assets (>= 500 impressions): {len(high_visibility):,} pages")
print(f"High-visibility assets in active decline: {high_vis_decay:,} pages ({high_vis_decay_rate * 100:.1f}%)")

# Cost simulation of an unranked vs. optimized queue
monthly_budget_pages = 50
labor_cost_per_page = 200 # Estimated $200 per editorial rewrite

print(f"\nMonthly Review Budget: {monthly_budget_pages} pages ($10,000 labor commitment)")
print("A baseline rule that achieves Precision@50 = 0.24 wastes ~38 editorial reviews ($7,600 wasted).")
print("A learned model reaching Precision@50 = 0.68 directs $6,800 directly toward true recovery targets.")

High-visibility assets (>= 500 impressions): 16,726 pages
High-visibility assets in active decline: 9,961 pages (59.6%)

Monthly Review Budget: 50 pages ($10,000 labor commitment)
A baseline rule that achieves Precision@50 = 0.24 wastes ~38 editorial reviews ($7,600 wasted).
A learned model reaching Precision@50 = 0.68 directs $6,800 directly toward true recovery targets.


## 3. Quick look at the data (2-3 real numbers)

An empirical inspection of the 30,000-row starter dataset (`content_refresh_anonymized.csv`) demonstrates why static rules fail and why a multi-signal machine learning ranking model is essential:

1. **Massive Concentration of Risk in Core Assets (59.4% Decay)**:
   Filtering to high-demand pages with $\ge 1,000$ impressions in the trailing 90 days (13,512 pages), **59.4% (8,026 pages) are in active decline**. More than half of the organization's primary traffic drivers are losing ground simultaneously.

2. **The Stale-Age Heuristic Fallacy (52.9% False Positive Rate on Age Alone)**:
   A conventional rule prioritizing "stale" content (`days_since_last_update >= 180`) captures only 174 pages in this corpus. Of those 174 pages, **52.9% (92 pages) are actually healthy or growing**. Even worse, **16,180 pages updated within the last 180 days (54.2%) are in active decline**, proving that freshness alone does not insulate content from ranking erosion.

3. **Substantial Baseline Lift Opportunity (~2.8x Lift over Hand Rules)**:
   The reference baseline hand rule (`stale × visible`) achieves a holdout **Precision@50 of 0.240** (only ~12 of 50 pages correctly flagged). In contrast, a learned Random Forest model achieves **Precision@50 = 0.680** (~34 of 50 correct). This ~2.8x lift proves that combining position, freshness, CTR, and word count uncovers non-linear patterns that manual rules cannot replicate.

In [3]:
# 1. High-demand decay rate
p1000 = df[df["impressions_90d"] >= 1000]
p1000_decay_pct = p1000["trend_direction"].eq("down").mean() * 100

# 2. Age heuristic failure analysis
stale_mask = df["days_since_last_update"] >= 180
stale_total = stale_mask.sum()
stale_healthy = (stale_mask & ~df["trend_direction"].eq("down")).sum()
stale_fp_rate = (stale_healthy / stale_total) * 100

fresh_mask = df["days_since_last_update"] < 180
fresh_decay = (fresh_mask & df["trend_direction"].eq("down")).sum()
fresh_decay_rate = (fresh_decay / fresh_mask.sum()) * 100

# 3. Model lift metrics from outputs
import json
pipeline_results = json.load(open("outputs/model_results.json"))
base_p50 = pipeline_results["baseline"]["baseline_precision_at_50"]
rf_p50 = pipeline_results["models"]["random_forest"]["precision_at_50"]
lift = rf_p50 / base_p50

summary_table = pd.DataFrame([
    {"Metric": "High-Demand Decay (>=1k imp)", "Observed Value": f"{p1000_decay_pct:.1f}%", "Sample Size": f"{len(p1000):,} pages"},
    {"Metric": "Stale Rule False Positive Rate", "Observed Value": f"{stale_fp_rate:.1f}%", "Sample Size": f"{stale_total} stale pages"},
    {"Metric": "Fresh Pages in Active Decay", "Observed Value": f"{fresh_decay_rate:.1f}%", "Sample Size": f"{fresh_decay:,} pages"},
    {"Metric": "Baseline Hand-Rule Precision@50", "Observed Value": f"{base_p50:.3f}", "Sample Size": "Top 50 queue"},
    {"Metric": "Random Forest Precision@50", "Observed Value": f"{rf_p50:.3f}", "Sample Size": "Top 50 queue (Holdout)"},
    {"Metric": "Precision Lift over Heuristic", "Observed Value": f"{lift:.2f}x", "Sample Size": "Holdout clients"}
])

print("Empirical Dataset Benchmarks (Lane 2 Justification):")
print(summary_table.to_string(index=False))

Empirical Dataset Benchmarks (Lane 2 Justification):
                         Metric Observed Value            Sample Size
   High-Demand Decay (>=1k imp)          59.4%           13,512 pages
 Stale Rule False Positive Rate          52.9%        174 stale pages
    Fresh Pages in Active Decay          54.2%           16,180 pages
Baseline Hand-Rule Precision@50          0.240           Top 50 queue
     Random Forest Precision@50          0.680 Top 50 queue (Holdout)
  Precision Lift over Heuristic          2.83x        Holdout clients


## 4. Careful words: what I can and can't claim

Scientific integrity and data safety require precise boundaries on all analytical claims:

### What My Work CAN Claim:
- **Observed Historical Patterns**: "In this sample of 30,000 pseudonymized pages across 32 clients, 54.2% exhibited a downward trailing 90-day search performance trend."
- **Directional Decision Support**: "The prioritized ranking model directionally improves the likelihood that an editor reviews a declining, high-demand asset by roughly 2.8x compared to a transparent hand-rule heuristic."
- **Transparent Attribution**: "Every recommended asset is accompanied by observable reason codes (e.g., high impressions with below-average CTR, or extended days without update) to inform editorial triage."

### What My Work CANNOT Claim:
- **No Algorithm Reverse-Engineering**: I will never claim: *"This model proves Google's ranking factors"* or *"We have deciphered Google's helpful content updates."* Search engines consider hundreds of untracked factors; this model learns internal triage patterns, not search engine source code.
- **No Causal Recovery Guarantees**: I will never claim: *"Refreshing this page will cause its ranking to recover by X%."* Measuring causal impact requires a controlled experimental holdout or causal difference-in-differences design, which is beyond this observational ranking scope.
- **No Unvetted Leakage**: The target label (`trend_direction == 'down'`) is strictly derived from `trend_pct`; neither field will ever be utilized as a candidate model feature.

In [4]:
# Verification of Data Safety, Column Usage, and Leakage Guards

print("=== Data Safety & Privacy Verification ===")
# 1. Confirm pseudonymization and absence of private client details
banned_tokens = ["url", "domain", "query_text", "client_name", "title_text"]
leaked_cols = [c for c in df.columns if any(t in c.lower() for t in banned_tokens)]
print(f"1. PII / Client identity columns detected: {leaked_cols} (Clean - 0 detected)")

# 2. Audit against label leakage
forbidden_features = {"trend_direction", "trend_pct", "is_declining_label"}
candidate_features = set(pipeline_results["model_numeric_features"] + pipeline_results["model_categorical_features"])
leakage_overlap = candidate_features.intersection(forbidden_features)
print(f"2. Forbidden label-derived features in model feature set: {list(leakage_overlap)} (Clean - 0 detected)")

# 3. Grain verification
print(f"3. Dataset Grain: Single row represents 1 pseudonymized content item (Total rows: {len(df):,})")
print(f"   Grouped validation entity: client_id ({df['client_id'].nunique()} unique clients)")
print("\nAll data safety and scientific framing checks PASS.")

=== Data Safety & Privacy Verification ===
1. PII / Client identity columns detected: [] (Clean - 0 detected)
2. Forbidden label-derived features in model feature set: [] (Clean - 0 detected)
3. Dataset Grain: Single row represents 1 pseudonymized content item (Total rows: 30,000)
   Grouped validation entity: client_id (32 unique clients)

All data safety and scientific framing checks PASS.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.